#DBU usage by resource group, last 7 days 

In [0]:
# Queries system.billing.usage — a real-time operational table, so unlike
# the historical VStone data itself, CURRENT_DATE()-relative windows here
# ARE correct: this is about monitoring actual compute spend happening
# now, not analyzing the frozen 2023-2024 dataset.
#
# SKU-name mapping adapted to VStone's actual pipeline/job names.


usage_by_resource = spark.sql("""
SELECT
    usage_date,
    sku_name,
    CASE
        WHEN sku_name LIKE '%DLT%' THEN 'DLT Pipelines (bronze/silver/gold)'
        WHEN sku_name LIKE '%JOBS%' THEN 'Jobs (chunking/bronze/silver/gold/day7)'
        ELSE 'Other'
    END AS resource_group,
    SUM(usage_quantity) AS total_dbus
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 7 DAYS
  AND usage_unit = 'DBU'
GROUP BY 1, 2, 3
ORDER BY 1 DESC
""")
display(usage_by_resource)

## Automated (Jobs/DLT) vs. interactive (all-purpose cluster) cost

In [0]:
automation_split = spark.sql("""
SELECT
    CASE
        WHEN sku_name LIKE '%ALL_PURPOSE%' THEN 'Manual / Interactive (expensive)'
        ELSE 'Automated Production (optimized)'
    END AS run_type,
    SUM(usage_quantity) AS total_dbus,
    ROUND((SUM(usage_quantity) / SUM(SUM(usage_quantity)) OVER()) * 100, 2) AS cost_percentage
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 30 DAYS
GROUP BY 1
""")
display(automation_split)

## DBU consumption trend by resource type, last 14 days

In [0]:
trend = spark.sql("""
SELECT
    usage_date,
    CASE
        WHEN sku_name LIKE '%DLT%' THEN 'DLT Pipelines'
        WHEN sku_name LIKE '%JOBS%' THEN 'Automated Jobs'
        ELSE 'Interactive/Manual'
    END AS usage_source,
    SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 14 DAYS
GROUP BY 1, 2
ORDER BY 1
""")
display(trend)

## Peak usage hours

In [0]:
peak_hours = spark.sql("""
SELECT
    hour(usage_start_time) AS usage_hour,
    dayofweek(usage_date) AS day_of_week,
    SUM(usage_quantity) AS dbus
FROM system.billing.usage
WHERE usage_date >= CURRENT_DATE() - INTERVAL 7 DAYS
GROUP BY 1, 2
ORDER BY 3 DESC
""")
display(peak_hours)

print("Resource usage analysis complete. See docs/day8_dashboard_setup.md")
print("for building these into a Lakeview dashboard via the UI.")